# 05 — Features (Phase 3)

Everything up to now produced *inputs*. This notebook produces the single table a
model can actually be trained on: **one row per flood station per 15 minutes**,
with everything knowable at that moment in the columns, and what happened next in
the label columns.

### The one rule

| | Time window | Lives in |
|---|---|---|
| **Features** | `(-inf, t]` — the past and the present | `features.py` |
| **Labels** | `(t, t+h]` — strictly the future | `labels.py` |

The windows must never touch. If they do, scores jump 20–30 points, the model
looks superb, and it fails on its first real day. Nothing in the output looks
wrong when this happens, which is why Part 3 checks it by *recomputing labels
from the raw 5-minute data* rather than by eyeballing the numbers.

### What this notebook decides

Part 4 answers a question we deliberately left open: **is the GFS rainfall
forecast worth including at all?** It has an awkward property — the archive
starts 23 March 2021, so the first cross-validation fold has no forecast data to
train on. Rather than build machinery to manage that and hope it was worth it, we
measure the feature's value first and only then decide.

### What you need to run

Notebooks 00–04 must have been run. This one writes ~350 MB to `data/features/`
and takes about 6 minutes.

## Setup

In [1]:
import os, sys, json, time
from pathlib import Path

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import numpy as np
import pandas as pd
pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 80)

from bkkflood.config import load_config
from bkkflood.rawio import connect
CFG = load_config()

# config lists years as [first, last]; expand it to every year in between.
_yrs = CFG["data"]["years"]
YEARS = list(range(_yrs[0], _yrs[-1] + 1)) if len(_yrs) == 2 else list(_yrs)

print("years:", YEARS, "| horizons:", CFG["horizons_hours"],
      "| tiers:", CFG["flood_event"]["tiers_cm"])

years: [2019, 2020, 2021, 2022, 2023, 2024, 2025] | horizons: [1, 3, 6] | tiers: {'nuisance': 5, 'advisory': 15, 'severe': 30}


## 1. Build the feature tables

One Parquet file per year. Each is roughly 3.5 million rows by 79 columns.

**Why this is one function call and not a page of pandas.** The joins happen
inside DuckDB and the result streams straight to disk. The first version of this
used `pandas.merge`, needed more than the 3.9 GB available, and was killed by the
operating system — and because the output was piped, an out-of-memory kill looked
exactly like success. It printed nothing and produced no file.

Roughly 50 seconds per year. Safe to re-run; it overwrites.

In [2]:
from bkkflood.features import write_feature_table

con = connect()
for year in YEARS:
    t0 = time.time()
    path = write_feature_table(year, con=con)
    size_mb = Path(path).stat().st_size / 1e6
    print(f"{year}  {size_mb:6.1f} MB  {time.time() - t0:5.1f}s  {path}")

2019    78.8 MB   43.0s  /Users/pritimmondal/Projects/bkk-flood-forecast/data/features/features_2019.parquet
2020    71.5 MB   45.7s  /Users/pritimmondal/Projects/bkk-flood-forecast/data/features/features_2020.parquet
2021    74.0 MB   47.9s  /Users/pritimmondal/Projects/bkk-flood-forecast/data/features/features_2021.parquet
2022    80.6 MB   49.4s  /Users/pritimmondal/Projects/bkk-flood-forecast/data/features/features_2022.parquet
2023    74.7 MB   55.9s  /Users/pritimmondal/Projects/bkk-flood-forecast/data/features/features_2023.parquet
2024    77.0 MB   58.9s  /Users/pritimmondal/Projects/bkk-flood-forecast/data/features/features_2024.parquet
2025    74.8 MB   60.9s  /Users/pritimmondal/Projects/bkk-flood-forecast/data/features/features_2025.parquet


## 2. What is in the table, and what is missing from it

Two things to look at.

**Coverage.** Every feature block should reach all 33 districts that have flood
sensors. When it does not, the cause is almost always a join key rather than
absent data — an earlier version of this notebook lost terrain for 16 of the 33
districts because the district lookup was inverted the wrong way round. It showed
up only as a suspiciously identical null rate appearing in two unrelated blocks
at once, which is the sort of thing you only notice if you look.

**Nulls.** Missing values are *not* filled in. A missing reading is not a dry
road. Filling it with zero teaches the model that broken sensors mean safety,
which is precisely backwards. LightGBM handles gaps natively.

In [3]:
import duckdb
from bkkflood.features import feature_columns

paths = [f"data/features/features_{y}.parquet" for y in YEARS]
q = duckdb.connect()

cols = [d[0] for d in q.execute(f"SELECT * FROM read_parquet({paths}) LIMIT 0").description]
feat = feature_columns(pd.DataFrame(columns=cols))
print(f"{len(cols)} columns total, of which {len(feat)} are model inputs\n")

null_sql = ", ".join(f'avg(CASE WHEN "{c}" IS NULL THEN 1.0 ELSE 0 END) AS "{c}"' for c in feat)
nulls = q.execute(f"SELECT {null_sql} FROM read_parquet({paths})").fetchdf().T[0]

block = pd.Series({c: c.split("_")[0] for c in feat})
summary = pd.DataFrame({"null_share": nulls.round(4), "block": block})
print(summary[summary.null_share > 0].sort_values("null_share", ascending=False).to_string())
print("\nfeatures with no missing values at all:", int((summary.null_share == 0).sum()))

80 columns total, of which 51 are model inputs

                       null_share  block
fl_hours_since_15cm        0.7346     fl
rain_antecedent_ratio      0.6367   rain
fl_hours_since_5cm         0.5077     fl
rain_fcst_1h               0.3053   rain
rain_fcst_3h               0.3053   rain
rain_fcst_6h               0.3053   rain
rain_x_recent_flood        0.0503   rain
fl_depth_lag_3h            0.0501     fl
fl_rise_1h                 0.0499     fl
fl_depth_lag_1h            0.0499     fl
fl_depth_now               0.0498     fl
fl_std_3h                  0.0498     fl
fl_mean_1h                 0.0498     fl
fl_max_3h                  0.0498     fl
fl_rise_15min              0.0498     fl
fl_max_24h                 0.0496     fl
rain_rf1hr_delta3h         0.0011   rain
rain_rf1hr_delta1h         0.0009   rain
rain_rf6hr_mean            0.0008   rain
rain_spread                0.0008   rain
rain_rf24hr_mean           0.0008   rain
rain_rf3hr_max             0.0008   rain
rain_rf3h

In [4]:
# Coverage: does every block actually reach every district and year?
cov = q.execute(f"""
SELECT date_part('year', ts)::INT AS year,
       count(DISTINCT station_code)                                   AS stations,
       count(DISTINCT district)                                       AS districts,
       round(avg(CASE WHEN rain_rf1hr_mean   IS NOT NULL THEN 1.0 ELSE 0 END), 3) AS rain,
       round(avg(CASE WHEN water_rise_1h_mean IS NOT NULL THEN 1.0 ELSE 0 END), 3) AS water,
       round(avg(CASE WHEN flow_mean          IS NOT NULL THEN 1.0 ELSE 0 END), 3) AS flow,
       round(avg(CASE WHEN rain_fcst_1h       IS NOT NULL THEN 1.0 ELSE 0 END), 3) AS gfs,
       round(avg(CASE WHEN terr_elev_m_p50    IS NOT NULL THEN 1.0 ELSE 0 END), 3) AS terrain,
       count(*) AS rows
FROM read_parquet({paths}) GROUP BY 1 ORDER BY 1
""").fetchdf()
print(cov.to_string(index=False))
print("\nThe gfs column is the one to read: GFS begins 23 March 2021, so 2019 and")
print("2020 are 0.000 and 2021 is partial. That is a real hole, not a bug, and")
print("Part 4 decides what to do about it.")

 year  stations  districts  rain  water  flow   gfs  terrain    rows
 2019        99         31 1.000    1.0   1.0 0.000      1.0 3468960
 2020        99         31 1.000    1.0   1.0 0.000      1.0 3478464
 2021       102         33 0.995    1.0   1.0 0.779      1.0 3531168
 2022       102         33 1.000    1.0   1.0 1.000      1.0 3574080
 2023       107         33 1.000    1.0   1.0 1.000      1.0 3749280
 2024       107         33 1.000    1.0   1.0 1.000      1.0 3759552
 2025       107         33 1.000    1.0   1.0 1.000      1.0 3749280

The gfs column is the one to read: GFS begins 23 March 2021, so 2019 and
2020 are 0.000 and 2021 is partial. That is a real hole, not a bug, and
Part 4 decides what to do about it.


### What is per-station and what is not

This is the most important limitation in the whole feature set, so it is printed
rather than buried in a document.

| Block | Resolution | Why |
|---|---|---|
| Flood history | **per station** | it is the sensor's own past |
| Rainfall | **per district** | rain and flood codes share a district prefix — 33/33 covered |
| Canal water level | **citywide** | canal codes name canals, not districts |
| Canal flow | **citywide** | same, and only 3 of 33 districts are reachable |
| Terrain | **per district** | no station coordinates exist |
| GFS forecast | ~13 km grid | coarser than a district |

Citywide features tell the model that the drainage network as a whole is under
stress. They can never say *which canal is backing up next to this road*. Fixing
this needs one spreadsheet from BMA — a station code and a latitude/longitude for
each of the 300 water-level and 30 flow sensors. It is the single highest-value
thing we can ask them for, and it is item 1 on the meeting list.

In [5]:
terr_gran = q.execute(f"SELECT DISTINCT terr_granularity FROM read_parquet({paths})").fetchdf()
print("terrain granularity marker in the table:", terr_gran.terr_granularity.tolist())
print("\nThis column exists so that no downstream report can describe the terrain")
print("features as per-station without contradicting a column in its own data.")

terrain granularity marker in the table: ['district']

This column exists so that no downstream report can describe the terrain
features as per-station without contradicting a column in its own data.


## 3. Do the features contain the future?

The failure that matters most, and the one that is invisible in a score.

Two checks, and both **recompute from the raw 5-minute data** rather than
reasoning about the numbers we already have. That distinction is not pedantry: on
four separate occasions in this project the *check* turned out to be wrong rather
than the data — once reading an absent archive as a difference, once comparing a
masked surface against itself, once using an absolute count where a density was
needed, and once mistaking ordinary autocorrelation for leakage. Every one of
those checks reasoned from summary statistics instead of going back to the source.

In [6]:
from bkkflood.features import check_features_against_raw
from bkkflood.labels import build_labels, check_labels_against_raw

probe_year = YEARS[-2]

fchk = check_features_against_raw(probe_year, con=con)
print("FEATURES —", json.dumps(fchk, indent=2))
assert fchk["passed"], "a feature window is looking forward — stop and fix it"

FEATURES — {
  "rows_compared": 250,
  "mismatches": 0,
  "rows_where_future_is_higher": 24,
  "rows_that_tracked_the_future": 0,
  "passed": true,
  "note": "fl_max_3h recomputed over (t-3h, t] from the 5-minute source"
}


In [7]:
lab = build_labels([probe_year], con=con)
lchk = check_labels_against_raw(lab, years=[probe_year], con=con)
print("LABELS —", json.dumps(lchk["checks"], indent=2), "\npassed:", lchk["passed"])
assert lchk["passed"], "labels disagree with the raw data"

from bkkflood.labels import label_summary
print("\n", label_summary(lab).to_string(index=False))
del lab

LABELS — [
  {
    "horizon_h": 1,
    "rows_compared": 567,
    "mismatches": 0,
    "agrees": true
  },
  {
    "horizon_h": 3,
    "rows_compared": 567,
    "mismatches": 0,
    "agrees": true
  },
  {
    "horizon_h": 6,
    "rows_compared": 567,
    "mismatches": 0,
    "agrees": true
  }
] 
passed: True

  tier_cm  horizon_h  rows_scorable  positives  base_rate  one_in  positives_onset  onset_share_of_positives
       5          1        3451112       2130   0.000617    1620             1327                     0.623
      15          1        3451112        377   0.000109    9154              251                     0.666
      30          1        3451112         43   0.000012   80258               28                     0.651
       5          3        3450729       4481   0.001299     770             3682                     0.822
      15          3        3450729        856   0.000248    4031              729                     0.852
      30          3        3450729     

**Reading the label summary.** `onset_share_of_positives` is the column that
matters. It is the fraction of positive rows where the road was *dry* at the
moment of the forecast — the rows that genuinely require forecasting rather than
remembering. It rises with the horizon, because over six hours there is more time
for a dry road to become a flooded one.

Every recall figure in this project is reported twice: overall, and on onset rows
alone. The previous version of this project published 55% recall, which turned
out to be ~100% on already-flooded rows and 9% on real onsets.

## 4. Is the GFS forecast worth keeping?

**The decision this notebook exists to make.** GFS is a global weather model at
roughly 13 km resolution. Bangkok is about 40 km across, so a handful of grid
cells cover the entire city, and Thai rainfall is convective — intense cells a
few kilometres wide. There is a real chance GFS simply cannot see the rain that
floods a particular road.

Including it is not free: the archive begins 23 March 2021, so the first
cross-validation fold trains on two years with no forecast at all. So we measure
first.

Three tests, each stricter than the last:

1. **Can it predict rain?** Compare the forecast for an hour against what the
   gauges recorded in that hour — on **wet hours only**. About 74% of hours are
   dry everywhere, so an overall agreement score mostly measures agreement about
   nothing happening.
2. **Does it separate floods from non-floods on its own?**
3. **Does it add anything we do not already have from the gauges?** — with a
   control that answers the obvious objection, below.

In [8]:
from bkkflood.features import forecast_value_test

fcst_years = [y for y in YEARS if y >= 2021]
res = forecast_value_test(fcst_years, tier_cm=15, horizon_h=3, con=con)

print("1. CAN GFS PREDICT BANGKOK RAIN, HOUR BY HOUR?")
print(res["meteorological_skill"].round(4).to_string(index=False))

1. CAN GFS PREDICT BANGKOK RAIN, HOUR BY HOUR?
 hours_compared  share_of_hours_dry_both_ways  wet_hours  wet_hour_correlation  wet_hour_hit_rate  mean_gauge_mm_when_wet  mean_fcst_mm_when_wet
        1377409                        0.7768     307484                0.0142             0.1556                  0.8353                 0.7723


**The answer is essentially no.** A wet-hour correlation near zero and a hit
rate well under half means that knowing GFS expects rain in a district in a given
hour tells you very little about whether rain actually falls there. This is the
expected result for a 13 km model over convective tropical rainfall, and it is
worth stating plainly rather than hoping nobody checks.

Which makes the next result surprising.

In [9]:
print("2 & 3. DOES IT HELP PREDICT FLOODS?  (PR-AUC — higher is better)")
print(res["value_against_label"].round(5).to_string(index=False))
print("\nVERDICT")
print(res["verdict"].T.to_string(header=False))

2 & 3. DOES IT HELP PREDICT FLOODS?  (PR-AUC — higher is better)
                           score  pr_auc_all  pr_auc_onset_only  positives_all  positives_onset  base_rate  lift_over_base_onset
                   gauge_past_3h     0.05121            0.00550          12279             9131    0.00072              10.29230
            gfs_forecast_next_3h     0.00512            0.00368          12279             9131    0.00072               6.89703
        gauge_past_plus_forecast     0.03892            0.00681          12279             9131    0.00072              12.74626
gauge_past_plus_SEASONAL_control     0.05201            0.00601          12279             9131    0.00072              11.26063
       current_depth (reference)     0.31238            0.03745          12279             9131    0.00072              70.11725

VERDICT
onset_pr_auc_gain_from_forecast                                   0.001311
onset_pr_auc_gain_from_seasonality_alone                          0.000517
ga

### Reading this table

**PR-AUC is not a percentage and its floor is not zero.** The no-skill value is
the base rate — about 1 positive in 930 rows here. A PR-AUC of 0.008 is roughly
ten times better than chance. The `lift_over_base_onset` column does that
division so nobody has to.

**`current_depth` is in the table as a reference point, not a candidate.** It
scores far above everything else, and that is the central difficulty of this
project rather than a good sign: the strongest available signal is "this road is
already wet", which is a monitor, not a forecast. On onset rows — where the road
is dry — it is much weaker.

**The control column is the point.** A forecast with near-zero hourly
correlation should not be able to improve anything. So before crediting it, we
replace every forecast value with the *average* forecast for that district,
month and hour — a series carrying all of the seasonality and none of the
day-to-day skill. If the gain is really just "GFS knows August is wet", it will
survive this substitution, and we should drop the feature: `cal_monsoon` and
`cal_doy_sin/cos` already provide seasonality, for free and with no gap in 2019–2020.

The verdict rows split the gain into the part a calendar could have provided and
the part it could not.

In [10]:
v = res["verdict"].iloc[0]
total = v["onset_pr_auc_gain_from_forecast"]
seasonal = v["onset_pr_auc_gain_from_seasonality_alone"]
real = v["gain_attributable_to_actual_forecast_skill"]

print(f"total onset gain from adding the forecast : {total:+.5f}")
print(f"  ...of which a calendar could explain    : {seasonal:+.5f}  ({seasonal/total:.0%})")
print(f"  ...genuine day-to-day forecast skill    : {real:+.5f}  ({real/total:.0%})")
print()
if real > 0.5 * total and real > 0:
    print("DECISION: keep rain_fcst_*. Most of the gain is not seasonality.")
    print("Conditions attached:")
    print("  - fold 1 (train 2019-2020) has no forecast data. Train it without")
    print("    these columns rather than imputing values that never existed.")
    print("  - re-run the ablation in Phase 5 on the real model. This test uses a")
    print("    sum of two rainfall numbers, not a trained model, and a gradient-")
    print("    boosted tree may extract more from it -- or less.")
else:
    print("DECISION: drop rain_fcst_*. The gain is seasonality, which the")
    print("calendar features already provide without a hole in 2019-2020.")

total onset gain from adding the forecast : +0.00131
  ...of which a calendar could explain    : +0.00052  (39%)
  ...genuine day-to-day forecast skill    : +0.00079  (61%)

DECISION: keep rain_fcst_*. Most of the gain is not seasonality.
Conditions attached:
  - fold 1 (train 2019-2020) has no forecast data. Train it without
    these columns rather than imputing values that never existed.
  - re-run the ablation in Phase 5 on the real model. This test uses a
    sum of two rainfall numbers, not a trained model, and a gradient-
    boosted tree may extract more from it -- or less.


### The honest caveat on `rain_fcst_*`

These features are built as the rain GFS expected over `(t, t+h]`, taken from
Open-Meteo's archived-forecast series. That series is stitched together from the
first hours of each successive model run — so the value at `t+6h` came from a run
launched a few hours before `t+6h`, which may be slightly *after* `t`.

At the 1-hour horizon this does not matter. At 6 hours it means the feature may
carry a little information that was not strictly available at the moment of the
forecast, which would make the 6-hour model look marginally better in testing
than in production.

This is written down rather than quietly accepted because it has the same shape
as a mistake this project already made once: Open-Meteo's forecast endpoint
silently served ERA5 reanalysis — *what actually fell* — and it passed every
naming convention we had while being the answer sheet. If Phase 5 shows the
6-hour model leaning on these columns, the fix is Open-Meteo's Previous Runs API,
which serves a fixed lead time instead of stitching.

## 5. The feature contract

A written record of what every column means, its resolution, and whether it is a
model input. Phase 7 builds an API from this; if training and serving disagree
about what `fl_std_3h` means, that is where it will surface.

In [11]:
RESOLUTION = {
    "fl": "per station", "rain": "per district", "water": "citywide",
    "flow": "citywide", "terr": "per district (no station coordinates)",
    "era5": "per district (13 km grid) — PAST rain, never a forecast",
    "cal": "per row", "tide": "per row (astronomical phase, NOT height)",
}
def resolution(c):
    if c.startswith("rain_fcst"): return "per district (13 km GFS grid) — FORECAST"
    return RESOLUTION.get(c.split("_")[0], "per row")

contract = pd.DataFrame({
    "column": cols,
    "role": ["label" if c.startswith("y_") else
             "evaluation only" if c.startswith("is_onset_") else
             "key" if c in ("station_code", "ts", "district", "district_code") else
             "metadata" if c == "terr_granularity" else "feature" for c in cols],
})
contract["resolution"] = contract.column.map(resolution)
contract.loc[contract.role != "feature", "resolution"] = ""
contract["null_share"] = contract.column.map(nulls.round(4)).fillna("")

out = Path("docs/reports/feature_contract.md")
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w") as f:
    f.write("# Feature contract — v3.0\n\n")
    f.write(f"Generated by `notebooks/05_features.ipynb`. Years {YEARS[0]}–{YEARS[-1]}, ")
    f.write(f"{len(cols)} columns, {len(feat)} model inputs.\n\n")
    f.write("`is_onset_*` is evaluation metadata and must never be a model input: it is\n")
    f.write("derived from the label window's starting condition.\n\n")
    f.write(contract.to_markdown(index=False))
    f.write("\n")
print("wrote", out)
print(contract.groupby("role").size().to_string())

wrote docs/reports/feature_contract.md
role
evaluation only     9
feature            51
key                 4
label              15
metadata            1


## What Phase 3 produced

- `data/features/features_YYYY.parquet` — 7 years, ~3.5 M rows each, 79 columns
- `docs/reports/feature_contract.md` — what every column means
- A measured, recorded decision on GFS forecast rain

**Next:** `06_baselines.ipynb` establishes what "good" means before any model is
trained. Baselines recorded afterwards are baselines chosen after seeing the
result, which is a different activity.